# Biohub Division Model Training V1

**Goal**: Train a 3D CNN classifier that predicts P(division) for candidate parent cells.
This learned probability replaces the heuristic `ILP_DIVISION_WEIGHT=0.78` with a
per-candidate score, giving the ILP solver real signal about which cells are about to divide.

**Why this matters**: 1 division = 74 edges in metric weight. A model that correctly
identifies 10 more divisions than the heuristic → +0.74 in weighted division score.

**Architecture**: 3D ResNet-18 (inflated 2D→3D) on spatio-temporal crops (t-1, t, t+1).

**Training data**: GT GEFF files from training embryos (44b6 + 6bba).
- Positive: cells with out_degree==2 in GT graph (division parents)
- Negative: cells with out_degree==1 (normal continuations), 10:1 ratio

**Output**: Saved model weights + calibration curve → uploaded as Kaggle dataset
for use in the integrated pipeline.

In [ ]:
import os
import sys
import time
import json
import math
import random
import pickle
import zipfile
import subprocess
from pathlib import Path
from collections import defaultdict

# ── Paths ─────────────────────────────────────────────────────────────────────
COMPETITION = 'biohub-cell-tracking-during-development'
COMP_DIR = next(
    (p for p in [
        Path(f'/kaggle/input/competitions/{COMPETITION}'),
        Path(f'/kaggle/input/{COMPETITION}'),
    ] if p.exists()),
    Path(f'/kaggle/input/{COMPETITION}')
)
TRAIN_DIR = COMP_DIR / 'train'
WORKING_DIR = Path('/kaggle/working')
OUTPUT_DIR = WORKING_DIR / 'division_model'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Config ────────────────────────────────────────────────────────────────────
CV_HOLDOUT_PREFIX = '6bba'   # embryo held out for validation (same as CV validator)
CROP_T = 3                   # frames: t-1, t, t+1
CROP_Z = 16                  # voxels in Z
CROP_Y = 48                  # voxels in Y
CROP_X = 48                  # voxels in X
NEG_POS_RATIO = 10           # negative:positive sampling ratio
BATCH_SIZE = 8
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
RANDOM_SEED = 42
MAX_VIDEOS = None            # set to integer to limit for debugging

random.seed(RANDOM_SEED)
print(f'TRAIN_DIR: {TRAIN_DIR}, exists: {TRAIN_DIR.exists()}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'CV hold-out: {CV_HOLDOUT_PREFIX}_* (same fold as CV validator)')

## Step 1: Install Dependencies

We need zarr, geff, and tracksdata from the pilkwang offline pack.

In [ ]:
# Install offline packages from pilkwang support pack
import importlib.util

SUPPORT_PACK_CANDIDATES = [
    Path('/kaggle/input/biohub-tracking-support-pack-50ep-v1'),
    Path('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1'),
]
SUPPORT_PACK = next((p for p in SUPPORT_PACK_CANDIDATES if p.exists()), None)
print(f'Support pack: {SUPPORT_PACK}')

def pkg_missing(name):
    return importlib.util.find_spec(name) is None

NEEDED = ['zarr', 'geff', 'tracksdata', 'blosc2', 'numcodecs']
missing = [n for n in NEEDED if pkg_missing(n)]
print(f'Missing packages: {missing}')

if missing and SUPPORT_PACK:
    wheel_dirs = [SUPPORT_PACK / 'wheels', SUPPORT_PACK]
    wheel_dir = next((d for d in wheel_dirs if d.exists()), None)
    if wheel_dir:
        whl_files = list(wheel_dir.glob('*.whl')) + list(wheel_dir.glob('*.tar.gz'))
        print(f'Found {len(whl_files)} wheel/tar files in {wheel_dir}')
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
             f'--find-links={wheel_dir}'] + missing,
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print('pip stderr:', result.stderr[-2000:])
        else:
            print('Installed offline:', missing)
    else:
        print('WARNING: wheel_dir not found, trying pip with internet...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing)

# Verify
for name in NEEDED:
    status = 'OK' if not pkg_missing(name) else 'MISSING'
    print(f'  {name}: {status}')

## Step 2: Extract Division Events from GT Lineage Graphs

Load each training GEFF file and extract:
- **Positive samples**: (video_stem, frame_t, z, y, x) for cells with out_degree==2 (division parent)
- **Negative samples**: (video_stem, frame_t, z, y, x) for cells with out_degree==1 (normal cell)

In [ ]:
import geff
import zarr
import numpy as np

def load_geff_graph(geff_path: Path):
    """Load a GEFF graph and return nodes and edges as plain dicts."""
    try:
        import tracksdata as td
        graph = td.TrackGraph.from_geff(str(geff_path))
        nodes = {}
        for row in graph.node_attrs().iter_rows(named=True):
            nid = int(row['node_id'])
            nodes[nid] = {
                'node_id': nid,
                't': int(row['t']),
                'z': float(row['z']),
                'y': float(row['y']),
                'x': float(row['x']),
            }
        edges = []
        for row in graph.edge_attrs().iter_rows(named=True):
            edges.append((int(row['source_id']), int(row['target_id'])))
        return nodes, edges
    except Exception as e:
        print(f'  ERROR loading {geff_path.name}: {e}')
        return {}, []

def extract_division_samples(geff_path: Path, video_stem: str):
    """Return (positives, negatives) as lists of (video_stem, t, z, y, x)."""
    nodes, edges = load_geff_graph(geff_path)
    if not nodes:
        return [], []

    # Compute out_degree for each node
    out_degree = defaultdict(int)
    for src, tgt in edges:
        out_degree[src] += 1

    positives = []
    negatives = []
    for nid, node in nodes.items():
        deg = out_degree.get(nid, 0)
        entry = (video_stem, node['t'], node['z'], node['y'], node['x'])
        if deg >= 2:
            positives.append(entry)
        elif deg == 1:
            negatives.append(entry)
        # deg==0: leaf node (last frame or isolated), skip

    return positives, negatives

# ── Find all training GEFF files ─────────────────────────────────────────────
all_geff_files = sorted(TRAIN_DIR.glob('*.geff')) if TRAIN_DIR.exists() else []
print(f'Found {len(all_geff_files)} GEFF files in {TRAIN_DIR}')

# Split into train / val
train_geff = [p for p in all_geff_files if not p.stem.startswith(CV_HOLDOUT_PREFIX)]
val_geff   = [p for p in all_geff_files if p.stem.startswith(CV_HOLDOUT_PREFIX)]
print(f'Train GEFFs: {len(train_geff)} | Val GEFFs (held out): {len(val_geff)}')

if MAX_VIDEOS is not None:
    train_geff = train_geff[:MAX_VIDEOS]
    print(f'DEBUG: limited to {MAX_VIDEOS} videos')

# ── Extract samples ───────────────────────────────────────────────────────────
train_pos, train_neg = [], []
val_pos, val_neg     = [], []

print('\nExtracting training samples...')
for geff_path in train_geff:
    pos, neg = extract_division_samples(geff_path, geff_path.stem)
    train_pos.extend(pos)
    train_neg.extend(neg)
    if pos:
        print(f'  {geff_path.stem}: {len(pos)} divisions, {len(neg)} normals')

print('\nExtracting validation samples...')
for geff_path in val_geff:
    pos, neg = extract_division_samples(geff_path, geff_path.stem)
    val_pos.extend(pos)
    val_neg.extend(neg)

print(f'\nTRAIN: {len(train_pos)} positives, {len(train_neg)} negatives')
print(f'VAL:   {len(val_pos)} positives, {len(val_neg)} negatives')
print(f'Positive rate (train): {len(train_pos) / (len(train_pos) + len(train_neg)):.3%}')

## Step 3: Build Dataset — Extract 3D+t Crops from Zarr Volumes

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Cache open Zarr stores to avoid repeated file opens
_ZARR_CACHE: dict = {}

def get_zarr_volume(video_stem: str):
    """Return the Zarr array for a training video (T, Z, Y, X)."""
    if video_stem in _ZARR_CACHE:
        return _ZARR_CACHE[video_stem]

    zarr_path = TRAIN_DIR / f'{video_stem}.zarr'
    if not zarr_path.exists():
        raise FileNotFoundError(f'Zarr not found: {zarr_path}')

    store = zarr.open(str(zarr_path), mode='r')
    # Competition zarr might have a nested structure — find the data array
    if hasattr(store, 'data'):
        arr = store.data
    elif isinstance(store, zarr.Array):
        arr = store
    else:
        # Try common nested paths
        for key in ['0', 'data', 'raw', 'image']:
            if key in store:
                arr = store[key]
                break
        else:
            # Just use the first child
            arr = next(iter(store.values()))

    _ZARR_CACHE[video_stem] = arr
    return arr


def extract_crop(video_stem: str, t: int, z: float, y: float, x: float,
                 crop_t=CROP_T, crop_z=CROP_Z, crop_y=CROP_Y, crop_x=CROP_X):
    """
    Extract a (crop_t, crop_z, crop_y, crop_x) crop centered on (t,z,y,x).
    Returns a float32 numpy array normalized to [0, 1].
    Returns None if the crop is out of bounds.
    """
    try:
        vol = get_zarr_volume(video_stem)
    except Exception:
        return None

    T, Z, Y, X = vol.shape[-4], vol.shape[-3], vol.shape[-2], vol.shape[-1]

    # Frame range: t-1, t, t+1
    t_start = t - (crop_t // 2)
    t_end   = t_start + crop_t

    # Spatial centers (integer voxel coords)
    zi, yi, xi = int(round(z)), int(round(y)), int(round(x))

    z0, z1 = zi - crop_z // 2, zi + crop_z // 2
    y0, y1 = yi - crop_y // 2, yi + crop_y // 2
    x0, x1 = xi - crop_x // 2, xi + crop_x // 2

    # Check bounds (require all frames to be valid)
    if t_start < 0 or t_end > T:
        return None
    if z0 < 0 or z1 > Z or y0 < 0 or y1 > Y or x0 < 0 or x1 > X:
        return None

    crop = np.array(vol[t_start:t_end, z0:z1, y0:y1, x0:x1], dtype=np.float32)

    # Normalize per-crop to [0, 1]
    vmin, vmax = crop.min(), crop.max()
    if vmax > vmin:
        crop = (crop - vmin) / (vmax - vmin)
    else:
        crop = np.zeros_like(crop)

    return crop  # shape: (crop_t, crop_z, crop_y, crop_x)


class DivisionDataset(Dataset):
    """Dataset of 3D+t crops labeled as division (1) or non-division (0)."""

    def __init__(self, positives, negatives, neg_pos_ratio=NEG_POS_RATIO,
                 augment=True, seed=RANDOM_SEED):
        rng = random.Random(seed)

        # Subsample negatives
        n_neg = min(len(negatives), len(positives) * neg_pos_ratio)
        sampled_neg = rng.sample(negatives, n_neg)

        self.samples = [(s, 1) for s in positives] + [(s, 0) for s in sampled_neg]
        rng.shuffle(self.samples)
        self.augment = augment
        self.n_pos = len(positives)
        self.n_neg = n_neg
        print(f'Dataset: {self.n_pos} pos + {self.n_neg} neg = {len(self.samples)} total')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        (video_stem, t, z, y, x), label = self.samples[idx]

        crop = extract_crop(video_stem, t, z, y, x)

        if crop is None:
            # Return zeros if crop fails (out of bounds)
            crop = np.zeros((CROP_T, CROP_Z, CROP_Y, CROP_X), dtype=np.float32)

        if self.augment:
            # Random flips in Y and X only (not Z — optical axis, not Z — preserve orientation)
            if random.random() < 0.5:
                crop = crop[:, :, ::-1, :].copy()   # flip Y
            if random.random() < 0.5:
                crop = crop[:, :, :, ::-1].copy()   # flip X
            # Random brightness jitter
            crop = (crop * (0.8 + 0.4 * random.random())).clip(0, 1)

        # Add channel dim: (1, T, Z, Y, X)
        tensor = torch.from_numpy(crop).unsqueeze(0)
        return tensor, torch.tensor(label, dtype=torch.float32)


print('Building training dataset...')
train_ds = DivisionDataset(train_pos, train_neg, augment=True)
print('Building validation dataset...')
val_ds   = DivisionDataset(val_pos, val_neg, augment=False, neg_pos_ratio=20)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

## Step 4: Define 3D ResNet-18 Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class ResBlock3D(nn.Module):
    """Basic 3D residual block."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, stride=1,      padding=1, bias=False)
        self.bn2   = nn.BatchNorm3d(out_ch)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm3d(out_ch),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class DivisionNet(nn.Module):
    """
    Lightweight 3D ResNet for division classification.
    Input:  (B, 1, T=3, Z=16, Y=48, X=48)
    Output: (B,) logit for P(division)
    """
    def __init__(self):
        super().__init__()
        # Stem — reduce spatial dims quickly
        self.stem = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=(1,3,3), stride=(1,2,2), padding=(0,1,1), bias=False),
            nn.BatchNorm3d(32),
            nn.ReLU(),
        )
        # Residual stages
        self.layer1 = ResBlock3D(32,  64, stride=(1,2,2))  # T=3, Z=8, Y=12, X=12
        self.layer2 = ResBlock3D(64, 128, stride=(1,2,2))  # T=3, Z=4, Y=6,  X=6
        self.layer3 = ResBlock3D(128,256, stride=(1,2,2))  # T=3, Z=2, Y=3,  X=3
        self.layer4 = ResBlock3D(256,256, stride=1)

        # Global average pool → FC → scalar
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256, 1)

    def forward(self, x):
        x = self.stem(x)    # (B, 32, T, Z/2, Y/2, X/2)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)   # (B, 256)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)  # (B,)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model = DivisionNet().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')

# Test forward pass
dummy = torch.zeros(2, 1, CROP_T, CROP_Z, CROP_Y, CROP_X, device=device)
out = model(dummy)
print(f'Forward pass test: input {tuple(dummy.shape)} -> output {tuple(out.shape)}')

## Step 5: Train

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Positive weight for BCEWithLogitsLoss: inverse of positive rate
pos_weight_val = NEG_POS_RATIO  # ~10x more negatives than positives
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_val], device=device)
)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

BEST_MODEL_PATH = OUTPUT_DIR / 'division_net_best.pt'
LAST_MODEL_PATH = OUTPUT_DIR / 'division_net_last.pt'
TRAIN_LOG_PATH  = OUTPUT_DIR / 'train_log.json'

train_log = []
best_val_auc = -1.0


def compute_metrics(loader, model, device, label='eval'):
    """Compute AUC and accuracy on a DataLoader."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(yb.cpu().numpy().tolist())

    from sklearn.metrics import roc_auc_score, average_precision_score
    import numpy as np
    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)

    # AUC and AP (handles class imbalance better)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = 0.0
    try:
        ap = average_precision_score(y_true, y_prob)
    except Exception:
        ap = 0.0

    preds = (y_prob > 0.5).astype(int)
    acc   = (preds == y_true).mean()
    tp = ((preds == 1) & (y_true == 1)).sum()
    fp = ((preds == 1) & (y_true == 0)).sum()
    fn = ((preds == 0) & (y_true == 1)).sum()
    prec  = tp / (tp + fp + 1e-9)
    rec   = tp / (tp + fn + 1e-9)
    print(f'  [{label}] AUC={auc:.4f}  AP={ap:.4f}  Acc={acc:.4f}  '
          f'Prec={prec:.4f}  Rec={rec:.4f}  TP={tp}  FP={fp}  FN={fn}')
    return auc, ap


print(f'Training for {NUM_EPOCHS} epochs...')
t0_all = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    t0 = time.time()

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_batches  += 1

    scheduler.step()
    avg_loss = epoch_loss / max(n_batches, 1)
    elapsed  = time.time() - t0
    print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  loss={avg_loss:.4f}  lr={scheduler.get_last_lr()[0]:.2e}  {elapsed:.0f}s')

    # Validate every 5 epochs
    if epoch % 5 == 0 or epoch == NUM_EPOCHS:
        val_auc, val_ap = compute_metrics(val_loader, model, device, label=f'val ep{epoch}')
        log_entry = {'epoch': epoch, 'train_loss': avg_loss, 'val_auc': val_auc, 'val_ap': val_ap}
        train_log.append(log_entry)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f'  -> New best AUC={val_auc:.4f}  saved to {BEST_MODEL_PATH}')

torch.save(model.state_dict(), LAST_MODEL_PATH)
TRAIN_LOG_PATH.write_text(json.dumps(train_log, indent=2))
print(f'\nTraining complete in {(time.time()-t0_all)/60:.1f} min')
print(f'Best val AUC: {best_val_auc:.4f}')

## Step 6: Calibrate with Isotonic Regression

Raw sigmoid outputs from the network may be poorly calibrated. We fit an isotonic
regression on the validation fold to map raw probabilities to calibrated ones.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
import numpy as np

# Load best model and get val probabilities
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

val_probs_raw, val_labels_cal = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs  = torch.sigmoid(logits).cpu().numpy()
        val_probs_raw.extend(probs.tolist())
        val_labels_cal.extend(yb.numpy().tolist())

y_prob_raw = np.array(val_probs_raw)
y_true_cal = np.array(val_labels_cal)

# Fit isotonic regression calibrator
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(y_prob_raw, y_true_cal)
y_prob_cal = calibrator.predict(y_prob_raw)

# Print calibration quality
n_bins = 10
fraction_of_positives, mean_predicted_value = calibration_curve(y_true_cal, y_prob_cal, n_bins=n_bins)
print('Calibration curve (after isotonic regression):')
print('  Mean predicted | Fraction positive')
for mp, fp in zip(mean_predicted_value, fraction_of_positives):
    print(f'  {mp:.3f}          | {fp:.3f}')

# Save calibrator
CAL_PATH = OUTPUT_DIR / 'calibrator.pkl'
with open(CAL_PATH, 'wb') as f:
    pickle.dump(calibrator, f)
print(f'\nCalibrator saved to {CAL_PATH}')

# Final AUC on calibrated probabilities
from sklearn.metrics import roc_auc_score, average_precision_score
final_auc = roc_auc_score(y_true_cal, y_prob_cal)
final_ap  = average_precision_score(y_true_cal, y_prob_cal)
print(f'Final calibrated AUC: {final_auc:.4f}  AP: {final_ap:.4f}')

## Step 7: Save All Outputs & Package as Dataset

Zip the model weights and calibrator for upload as a Kaggle dataset,
so the integrated pipeline (Notebook 4) can load them offline.

In [ ]:
# Save metadata
METADATA_PATH = OUTPUT_DIR / 'division_model_metadata.json'
metadata = {
    'crop_t':        CROP_T,
    'crop_z':        CROP_Z,
    'crop_y':        CROP_Y,
    'crop_x':        CROP_X,
    'cv_holdout':    CV_HOLDOUT_PREFIX,
    'best_val_auc':  best_val_auc,
    'final_cal_auc': final_auc,
    'final_cal_ap':  final_ap,
    'train_positives': len(train_pos),
    'train_negatives': len(train_neg),
    'val_positives':   len(val_pos),
    'val_negatives':   len(val_neg),
    'neg_pos_ratio':   NEG_POS_RATIO,
    'num_epochs':      NUM_EPOCHS,
    'architecture':    'DivisionNet_3DResNet18',
    'train_log':       train_log,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2))
print(f'Metadata saved to {METADATA_PATH}')

# Zip everything for dataset upload
ZIP_PATH = WORKING_DIR / 'biohub_division_model_v1.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR.iterdir():
        zf.write(f, f.name)
        print(f'  Added: {f.name} ({f.stat().st_size/1e6:.2f} MB)')

print(f'\nPackage: {ZIP_PATH} ({ZIP_PATH.stat().st_size/1e6:.2f} MB)')
print('\nNext steps:')
print('  1. Download this notebook output')
print('  2. Upload biohub_division_model_v1.zip as a Kaggle dataset')
print('  3. Add it as a data source to Notebook 4 (integrated pipeline)')
print('  4. Compare CV score with/without division model')

# Print final summary
print('\n' + '='*60)
print('DIVISION MODEL TRAINING SUMMARY')
print('='*60)
print(f'  Architecture: DivisionNet (3D ResNet-18 style)')
print(f'  Input:        (1, {CROP_T}, {CROP_Z}, {CROP_Y}, {CROP_X})  [C, T, Z, Y, X]')
print(f'  Training:     {len(train_pos)} positives + {len(train_neg)} negatives (sample {NEG_POS_RATIO}:1)')
print(f'  Val fold:     {CV_HOLDOUT_PREFIX}_* ({len(val_pos)} pos + {len(val_neg)} neg)')
print(f'  Best val AUC: {best_val_auc:.4f}')
print(f'  Cal. AUC:     {final_auc:.4f}')
print(f'  Cal. AP:      {final_ap:.4f}')